In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!unzip -q /content/drive/MyDrive/test.zip -d /content/
!unzip -q /content/drive/MyDrive/train.zip -d /content/

replace /content/test/IMG_146695.JPEG? [y]es, [n]o, [A]ll, [N]one, [r]ename: y


In [ ]:
import torch
from torch import nn
import torch.nn.functional as F


class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, kernel_size=3, use_res_connection=True) -> None:
        super().__init__()
        padding = (kernel_size - 1) // 2

        self.conv1 = nn.Conv2d(
            in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, padding=padding, stride=stride, bias=False
        )
        self.b1 = nn.BatchNorm2d(num_features=out_channels)

        self.conv2 = nn.Conv2d(
            in_channels=out_channels, out_channels=out_channels, kernel_size=kernel_size, padding=padding, bias=False
        )
        self.b2 = nn.BatchNorm2d(num_features=out_channels)

        if use_res_connection:
            self.res_mapping = nn.Sequential()
            if stride != 1 or in_channels != out_channels:
                self.res_mapping = nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                    nn.BatchNorm2d(out_channels)
                )
        else:
            self.res_mapping = None

    def forward(self, x):
        x_res = self.res_mapping(x) if self.res_mapping is not None else None

        x = F.relu(self.b1(self.conv1(x)))
        x = self.b2(self.conv2(x))

        if x_res is not None:
            x += x_res

        x = F.relu(x)
        return x

# Experiment 1 - conv channel widths too large / too small
# Experiment 2 - too few / too many conv channels
# Eksperyment 2 - bez res connection
class CNNImageClassifier(nn.Module):
    def __init__(self, in_channels: int, n_classes: int, conv_channels=None, use_res_connection=True) -> None:
        super().__init__()
        if conv_channels is None:
            conv_channels = [16, 32, 64, 128]

        self.initial_conv = nn.Sequential(
            nn.Conv2d(in_channels, conv_channels[0], kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(conv_channels[0]),
            nn.ReLU()
        )

        blocks = []
        in_ch = conv_channels[0]

        for out_ch in conv_channels:
            stride = 1 if in_ch == out_ch else 2
            blocks.append(ConvBlock(in_channels=in_ch, out_channels=out_ch, stride=stride, use_res_connection=use_res_connection))
            blocks.append(ConvBlock(in_channels=out_ch, out_channels=out_ch, stride=1, use_res_connection=use_res_connection))
            in_ch = out_ch

        self.conv_blocks = nn.Sequential(*blocks)

        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc_layers = nn.Sequential(
           nn.Linear(conv_channels[-1], 256),
           nn.ReLU(),
           nn.Dropout(p=0.5),
           nn.Linear(256, n_classes)
        )

    def forward(self, x):
        x = self.initial_conv(x)
        x = self.conv_blocks(x)
        x = self.avg_pool(x)
        x = torch.flatten(x, 1)
        x = self.fc_layers(x)
        return x


In [ ]:
import torch
import torchvision
from torchvision.transforms import transforms
from torch.utils.data import DataLoader, random_split, Dataset
from collections import Counter


class ImageClassifierDataset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, index):
        x, y = self.subset[index]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)


def get_dataloaders(train_transform, mean, std, image_size, data_dir="train/", batch_size=32, seed=42):
    initial_dataset = torchvision.datasets.ImageFolder(data_dir)

    # EDA showed unbalanced classes
    class_counts = Counter(initial_dataset.targets)
    num_classes = len(initial_dataset.classes)
    total_samples = sum(class_counts.values())

    class_weights = [total_samples / (num_classes * class_counts[i]) for i in range(num_classes)]
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

    # Transformations, datasets and dataloaders
    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    train_size = int(0.8 * len(initial_dataset))
    val_size = len(initial_dataset) - train_size
    generator = torch.Generator().manual_seed(seed)
    train_subset, val_subset = random_split(initial_dataset, [train_size, val_size], generator=generator)

    train_dataset = ImageClassifierDataset(train_subset, train_transform)
    val_dataset = ImageClassifierDataset(val_subset, val_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    return train_loader, val_loader, class_weights_tensor, initial_dataset.class_to_idx


In [ ]:
import torch
import os
from datetime import datetime
import torch.nn as nn
import torch.optim as optim
from torchvision.transforms import transforms

BATCH_SIZE = 64
EPOCHS = 50
LR = 0.001  # AdamW
# mean, std and image size are from EDA
MEAN = [0.5209, 0.4955, 0.4384]
STD = [0.2111, 0.2103, 0.2101]
IMAGE_SIZE = 64


def train(train_transform, conv_channels, use_res_connection):
    os.makedirs("models", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    model_save_path = os.path.join("models", f"model-{timestamp}.pt")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    train_loader, val_loader, class_weights, class_to_idx = get_dataloaders(
        train_transform, MEAN, STD, IMAGE_SIZE, data_dir="/content/train/", batch_size=BATCH_SIZE
    )
    class_weights = class_weights.to(device)

    model = CNNImageClassifier(
        in_channels=3,
        n_classes=50,
        conv_channels=conv_channels,
        use_res_connection=use_res_connection
    ).to(device)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=LR)

    best_val_acc = 0.0

    print("Training started")
    for epoch in range(EPOCHS):
        # --- TRENING ---
        model.train()
        running_loss = 0.0
        correct_train = 0
        total_train = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

        train_acc = correct_train / total_train
        train_loss_avg = running_loss / len(train_loader)

        # --- WALIDACJA ---
        model.eval()
        val_loss = 0.0
        correct_val = 0
        total_val = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        val_acc = correct_val / total_val
        val_loss_avg = val_loss / len(val_loader)

        print(f"Epoch [{epoch+1:02d}/{EPOCHS}] | "
              f"Train Loss: {train_loss_avg:.4f} - Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss_avg:.4f} - Acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), model_save_path)
            print("Accurancy improved, saved new model")

    print(f"\nTraining finished and model saved. The best val acc: {best_val_acc:.2f}%")

In [ ]:
import torch.backends.cudnn as cudnn
cudnn.benchmark = True

In [ ]:
train_transform = transforms.Compose([
transforms.RandomCrop(IMAGE_SIZE, padding=4),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(0.6)
])
conv_channels = [32, 64, 128, 256]
use_res_connection = True

train(train_transform, conv_channels, use_res_connection)

Training started
Epoch [01/50] | Train Loss: 3.4111 - Acc: 0.11% | Val Loss: 3.2257 - Acc: 0.14%
Accurancy improved, saved new model
Epoch [02/50] | Train Loss: 3.1135 - Acc: 0.17% | Val Loss: 2.9026 - Acc: 0.21%
Accurancy improved, saved new model
Epoch [03/50] | Train Loss: 2.9447 - Acc: 0.20% | Val Loss: 2.7165 - Acc: 0.26%
Accurancy improved, saved new model
Epoch [04/50] | Train Loss: 2.7882 - Acc: 0.24% | Val Loss: 2.5756 - Acc: 0.28%
Accurancy improved, saved new model
Epoch [05/50] | Train Loss: 2.6599 - Acc: 0.27% | Val Loss: 2.3469 - Acc: 0.34%
Accurancy improved, saved new model
Epoch [06/50] | Train Loss: 2.5428 - Acc: 0.30% | Val Loss: 2.4036 - Acc: 0.34%
Epoch [07/50] | Train Loss: 2.4334 - Acc: 0.33% | Val Loss: 2.1520 - Acc: 0.40%
Accurancy improved, saved new model
Epoch [08/50] | Train Loss: 2.3405 - Acc: 0.36% | Val Loss: 2.0461 - Acc: 0.43%
Accurancy improved, saved new model
Epoch [09/50] | Train Loss: 2.2469 - Acc: 0.38% | Val Loss: 2.0528 - Acc: 0.43%
Accurancy i

In [ ]:
import torch
import os
import csv
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Network parameters matching train.py
MEAN = [0.5209, 0.4955, 0.4384]
STD = [0.2111, 0.2103, 0.2101]
CONV_CHANNELS = [32, 64, 128, 256]
USE_RES_CONNECTION = True
N_CLASSES = 50

class TestDataset(Dataset):
    """
    Ładuje zbiór testowy, zachowując oryginalne nazwy plików.
    Operuje na bibliotece PIL, dokładnie tak jak wbudowany ImageFolder.
    """
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_files = [
            f for f in os.listdir(test_dir)
            if f.endswith(('.JPEG', '.jpeg', '.jpg', '.png'))
        ]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.test_dir, img_name)

        # Converting to RGB guarantees a consistent tensor shape
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, img_name

def generate_predictions(model_path, test_dir, output_csv):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Używane urządzenie: {device}")

    # The test transform is identical to val_transform
    test_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ])

    test_dataset = TestDataset(test_dir=test_dir, transform=test_transform)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    model = CNNImageClassifier(
        in_channels=3,
        n_classes=N_CLASSES,
        conv_channels=CONV_CHANNELS,
        use_res_connection=USE_RES_CONNECTION
    )

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    predictions = []

    print("Rozpoczęto generowanie predykcji...")
    with torch.no_grad():
        for images, filenames in test_loader:
            images = images.to(device)
            outputs = model(images)

            # Index of the class with the highest probability (0-49)
            _, predicted_indices = torch.max(outputs, 1)

            for filename, predicted_idx in zip(filenames, predicted_indices):
                # Append the filename and the predicted class straight to the list
                predictions.append([filename, predicted_idx.item()])

    with open(output_csv, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(predictions)

    print(f"Zapisano {len(predictions)} predykcji do pliku: {output_csv}")

if __name__ == '__main__':
    # Set the path to the model weights (.pt) before running
    MODEL_PATH = "model-20260415_181017.pt"
    TEST_DIR = "test/"
    OUTPUT_CSV = "pred.csv"

    if not os.path.exists(MODEL_PATH):
        print(f"Błąd: Plik modelu '{MODEL_PATH}' nie istnieje.")
    else:
        generate_predictions(MODEL_PATH, TEST_DIR, OUTPUT_CSV)

Używane urządzenie: cuda
Rozpoczęto generowanie predykcji...
Zapisano 10000 predykcji do pliku: pred.csv


In [ ]:
import torch
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader

# Parameters matching training
MEAN = [0.5209, 0.4955, 0.4384]
STD = [0.2111, 0.2103, 0.2101]
CONV_CHANNELS = [32, 64, 128, 256]
USE_RES_CONNECTION = True
N_CLASSES = 50
IMAGE_SIZE = 64

def test_on_train_set(model_path, data_dir, batch_size=64):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Używane urządzenie: {device}")

    # Exactly the same transform as the validation one in train.py
    # Resize(64, 64) added to rule out varying image sizes as a problem
    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ])

    # ImageFolder gives the network exactly the same class indices as during training
    dataset = torchvision.datasets.ImageFolder(root=data_dir, transform=transform)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Model initialisation
    model = CNNImageClassifier(
        in_channels=3,
        n_classes=N_CLASSES,
        conv_channels=CONV_CHANNELS,
        use_res_connection=USE_RES_CONNECTION
    )

    # Load the weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    print(f"Rozpoczęto ewaluację na zbiorze: {data_dir} (Liczba próbek: {len(dataset)})")

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    print(f"Dokładność (Accuracy) modelu na całym zbiorze: {accuracy * 100:.2f}%")

    # Sanity check - print the first 5 classes and their indices
    # print("\n--- Class mapping generated by PyTorch (ImageFolder) ---")
    # for i, (class_name, class_idx) in enumerate(dataset.class_to_idx.items()):
    #     if i < 5:
    #         print(f"Indeks PyTorch: {class_idx} -> Folder: {class_name}")
    # print("...")

if __name__ == '__main__':
    # Paths to the model and the extracted train folder
    MODEL_PATH = "model-20260415_181017.pt"
    TRAIN_DIR = "/content/train/"

    test_on_train_set(MODEL_PATH, TRAIN_DIR)

Używane urządzenie: cuda
Rozpoczęto ewaluację na zbiorze: /content/train/ (Liczba próbek: 88011)
Dokładność (Accuracy) modelu na całym zbiorze: 82.22%


In [ ]:
import torch
import os
import csv
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Network parameters matching train.py
MEAN = [0.5209, 0.4955, 0.4384]
STD = [0.2111, 0.2103, 0.2101]
CONV_CHANNELS = [32, 64, 128, 256]
USE_RES_CONNECTION = True
N_CLASSES = 50
IMAGE_SIZE = 64  # Zdefiniowany rozmiar obrazu

class TestDataset(Dataset):
    """
    Ładuje zbiór testowy, zachowując oryginalne nazwy plików.
    Operuje na bibliotece PIL, dokładnie tak jak wbudowany ImageFolder.
    """
    def __init__(self, test_dir, transform=None):
        self.test_dir = test_dir
        self.transform = transform
        self.image_files = [
            f for f in os.listdir(test_dir)
            if f.endswith(('.JPEG', '.jpeg', '.jpg', '.png'))
        ]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.test_dir, img_name)

        # Converting to RGB guarantees a consistent tensor shape
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        return image, img_name

def generate_predictions(model_path, test_dir, output_csv):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Używane urządzenie: {device}")

    # Key change: Resize applied before converting to a tensor
    test_transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(MEAN, STD)
    ])

    test_dataset = TestDataset(test_dir=test_dir, transform=test_transform)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    model = CNNImageClassifier(
        in_channels=3,
        n_classes=N_CLASSES,
        conv_channels=CONV_CHANNELS,
        use_res_connection=USE_RES_CONNECTION
    )

    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()

    predictions = []

    print("Rozpoczęto generowanie predykcji...")
    with torch.no_grad():
        for images, filenames in test_loader:
            images = images.to(device)
            outputs = model(images)

            # Index of the class with the highest probability (0-49)
            _, predicted_indices = torch.max(outputs, 1)

            for filename, predicted_idx in zip(filenames, predicted_indices):
                # Append the filename and the predicted class straight to the list
                predictions.append([filename, predicted_idx.item()])

    with open(output_csv, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(predictions)

    print(f"Zapisano {len(predictions)} predykcji do pliku: {output_csv}")

if __name__ == '__main__':
    # Set the path to the model weights (.pt) before running
    MODEL_PATH = "model-20260415_181017.pt"
    TEST_DIR = "test/"
    OUTPUT_CSV = "pred.csv"

    if not os.path.exists(MODEL_PATH):
        print(f"Błąd: Plik modelu '{MODEL_PATH}' nie istnieje.")
    else:
        generate_predictions(MODEL_PATH, TEST_DIR, OUTPUT_CSV)

Używane urządzenie: cuda
Rozpoczęto generowanie predykcji...
Zapisano 10000 predykcji do pliku: pred.csv


In [ ]:
!pip install transformers

In [ ]:
import torch
import os
import csv
from PIL import Image
from transformers import CLIPProcessor, CLIPModel
from torch.utils.data import Dataset, DataLoader

class TestDataset(Dataset):
    def __init__(self, test_dir):
        self.test_dir = test_dir
        self.image_files = [f for f in os.listdir(test_dir) if f.endswith(('.JPEG', '.jpeg', '.jpg', '.png'))]

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.test_dir, img_name)
        image = Image.open(img_path).convert('RGB')
        return image, img_name

def custom_collate_fn(batch):
    # 'batch' to lista krotek, np. [(Obraz1, "plik1.jpg"), (Obraz2, "plik2.jpg")]
    # Rozdzielamy to na dwie osobne listy:
    images = [item[0] for item in batch]
    filenames = [item[1] for item in batch]
    return images, filenames

def create_pseudo_ground_truth_clip(test_dir, train_dir, output_csv):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Generowanie Pseudo Ground Truth za pomocą CLIP na: {device}")

    class_names = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
    text_prompts = [f"a photo of a {name.replace('_', ' ')}" for name in class_names]

    print("Ładowanie modelu CLIP (to może potrwać chwilę)...")
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    test_dataset = TestDataset(test_dir=test_dir)

    # KLUCZOWA ZMIANA: Dodajemy collate_fn=custom_collate_fn
    test_loader = DataLoader(
        test_dataset,
        batch_size=32,
        shuffle=False,
        collate_fn=custom_collate_fn
    )

    predictions = []

    print("Rozpoczęto generowanie pseudo-etykiet (Sędzia ocenia test)...")
    with torch.no_grad():
        for images, filenames in test_loader:
            # images is now a plain list of PIL.Image objects, which the CLIP processor handles directly
            inputs = processor(text=text_prompts, images=images, return_tensors="pt", padding=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}

            outputs = model(**inputs)
            logits_per_image = outputs.logits_per_image
            probs = logits_per_image.softmax(dim=1)

            _, predicted_indices = torch.max(probs, 1)

            for filename, predicted_idx in zip(filenames, predicted_indices):
                predictions.append([filename, predicted_idx.item()])

    with open(output_csv, mode='w', newline='') as file:
        writer = csv.writer(file)
        writer.writerows(predictions)

    print(f"Sędzia (CLIP) zapisał {len(predictions)} predykcji do pliku: {output_csv}\n")

def calculate_accuracy(pseudo_gt_csv, model_csv, model_name):
    gt_dict = {}
    with open(pseudo_gt_csv, mode='r') as file:
        reader = csv.reader(file)
        for row in reader:
            if len(row) >= 2:
                gt_dict[row[0]] = str(row[1])

    correct = 0
    total = 0

    if not os.path.exists(model_csv):
        print(f"Błąd: Nie znaleziono pliku '{model_csv}'")
        return

    with open(model_csv, mode='r') as file:
        reader = csv.reader(file)
        for row in reader:
            if len(row) >= 2:
                filename = row[0]
                prediction = str(row[1])

                if filename in gt_dict:
                    total += 1
                    if gt_dict[filename] == prediction:
                        correct += 1

    if total > 0:
        accuracy = (correct / total) * 100
        print(f"Zgodność '{model_name}' z oceną Sędziego: {accuracy:.2f}% ({correct}/{total})")
    else:
        print(f"Brak wspólnych plików dla {model_name}.")

if __name__ == '__main__':
    TEST_DIR = "test/"
    TRAIN_DIR = "train/" # Potrzebne do odczytania nazw 50 klas!

    PSEUDO_GT_CSV = "clip_pseudo_ground_truth.csv"
    MODEL_1_CSV = "pred.csv" # Wygenerowane przez Twój skrypt z Resize(64,64)
    MODEL_2_CSV = "pred(5).csv" # Wygenerowane przez drugi model

    # 1. Run the judge (CLIP)
    create_pseudo_ground_truth_clip(TEST_DIR, TRAIN_DIR, PSEUDO_GT_CSV)

    # 2. Check which of the models agrees with CLIP most
    print("\n--- PORÓWNANIE MODELI ---")
    calculate_accuracy(PSEUDO_GT_CSV, MODEL_1_CSV, "Mój Model 1 (lepsze val acc)")
    calculate_accuracy(PSEUDO_GT_CSV, MODEL_2_CSV, "Mój Model 2 (gorsze val acc)")

Generowanie Pseudo Ground Truth za pomocą CLIP na: cuda
Ładowanie modelu CLIP (to może potrwać chwilę)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Rozpoczęto generowanie pseudo-etykiet (Sędzia ocenia test)...
Sędzia (CLIP) zapisał 10000 predykcji do pliku: clip_pseudo_ground_truth.csv


--- PORÓWNANIE MODELI ---
Zgodność 'Mój Model 1 (lepsze val acc)' z oceną Sędziego: 56.11% (5611/10000)
Zgodność 'Mój Model 2 (gorsze val acc)' z oceną Sędziego: 52.10% (5210/10000)
